In [ ]:
import pandas as pd
import json
from functions_and_classes.treatise_reference_data import treatise_paragraph_list

In [ ]:
from functions_and_classes.paper_processing import chapterScores

In [ ]:
from functions_and_classes.classes import *
from functions_and_classes.paper_io import p_to_dict, p_from_dict, c_to_dict, c_from_dict

In [ ]:
from data.jsons import json_list
import re

In [ ]:
a_w_p_reverse = {}
s_w_p_reverse = {}

In [ ]:
for para in treatise_paragraph_list:
    a_w_p_reverse[para] = 0
    s_w_p_reverse[para] = 0
print(len(a_w_p_reverse.keys()))
print(len(s_w_p_reverse.keys()))

In [ ]:
a_w_c_reverse = chapterScores(a_w_p_reverse)
s_w_c_reverse = chapterScores(s_w_p_reverse)
print(len(a_w_c_reverse.keys()))
print(len(s_w_c_reverse.keys()))

In [ ]:
awprev_df = pd.DataFrame.from_dict(a_w_p_reverse, orient='index', columns=['Blank'])
awcrev_df = pd.DataFrame.from_dict(a_w_c_reverse, orient='index', columns=['Blank'])
swprev_df = pd.DataFrame.from_dict(s_w_p_reverse, orient='index', columns=['Blank'])
swcrev_df = pd.DataFrame.from_dict(s_w_c_reverse, orient='index', columns=['Blank'])

In [ ]:
#generate the blanks
awprev_df = pd.DataFrame.from_dict(a_w_p_reverse, orient='index', columns=['Blank'])
awcrev_df = pd.DataFrame.from_dict(a_w_c_reverse, orient='index', columns=['Blank'])
swprev_df = pd.DataFrame.from_dict(s_w_p_reverse, orient='index', columns=['Blank'])
swcrev_df = pd.DataFrame.from_dict(s_w_c_reverse, orient='index', columns=['Blank'])
#fill them with the article data
for file in json_list:
    #create the paper object
    jf = open('data/jsons/'+file, 'r')
    jd = json.load(jf)
    jf.close()
    pawp = pd.DataFrame.from_dict(jd['a_w_p'], orient='index', columns=[file[:-5]])
    awprev_df[file[:-5]] = pawp[file[:-5]]
    pawc = pd.DataFrame.from_dict(jd['a_w_c'], orient='index', columns=[file[:-5]])
    awcrev_df[file[:-5]] = pawc[file[:-5]]
    pswp = pd.DataFrame.from_dict(jd['s_w_p'], orient='index', columns=[file[:-5]])
    swprev_df[file[:-5]] = pswp[file[:-5]]
    pswc = pd.DataFrame.from_dict(jd['s_w_c'], orient='index', columns=[file[:-5]])
    swcrev_df[file[:-5]] = pswc[file[:-5]]
#delete the blanks
del awprev_df['Blank']
del awcrev_df['Blank']
del swprev_df['Blank']
del swcrev_df['Blank']
#generate csvs
awprev_df.to_csv('data/awpreverse.csv')
awcrev_df.to_csv('data/awcreverse.csv')
swprev_df.to_csv('data/swpreverse.csv')
swcrev_df.to_csv('data/swcreverse.csv')

In [ ]:
#now let's create usable dictionaries 
awc111rev = awcrev_df.loc['1.1.1']
zfilter = awc111rev > 0
awc111revsc = awc111rev[zfilter].sort_values(ascending=False)
awc111revsc

In [ ]:


def locationFrequency(location, df):
    location_series = df.loc[location]
    zfilter = location_series > 0
    csloc_series = location_series[zfilter].sort_values(ascending=False)
    
    out_list = []
    
    for article in csloc_series.index:
        out_list.append((article,csloc_series.loc[article]))

    return out_list



In [ ]:
locationFrequency('Abs', awcrev_df)

In [ ]:
#To select a paragraph or chapter, it'll help to have selection menus to work with. Book, Section, Chapter, Paragraph.
#I'll have to do some work to narrow those down.
#Set it up such that you must select at least a chapter but if you don't select a paragraph you'll just get the list for the whole chapter.

In [ ]:
#Build a dictionary 
loc_dict = {}
books = ['Abstract', 'Introduction', 'I', 'II', 'III', 'Appendix']
sections = []
chapters = []
paragraphs = []

In [ ]:
book_list=[]
for loc in treatise_paragraph_list:
    structure_loc = loc.split('.')
   #create a book list:
    if structure_loc[0] not in book_list:
        book_list.append(structure_loc[0])

In [ ]:
chap_list = []
book_sect_dict = {}
for book in book_list:
    section_list = []
    for loc in treatise_paragraph_list:
        struct_loc = loc.split('.')
        if struct_loc[0] == book and struct_loc[1] not in section_list:
            section_list.append(struct_loc[1])
    book_sect_dict[book] = section_list

In [ ]:
book_sect_dict['0']

In [ ]:
import re

In [ ]:
chap_list = []
chapter_pattern = re.compile(r'(Abs)|(App\.)|(0)|(\d\.\d\.\d{1,2})')
for para in treatise_paragraph_list:
    chap_match = chapter_pattern.match(para).group()
    if chap_match not in chap_list:
        chap_list.append(chap_match)
chap_list

In [ ]:
section_list = []
section_pattern = re.compile(r'\d\.\d{1,2}')
for chapter in chap_list:
    sect_match = section_pattern.match(chapter)
    if sect_match != None:
        if sect_match.group() not in section_list:
            section_list.append(sect_match.group())
section_list

In [ ]:
#Intro case: 0.x
#regular books: X.x.x.x 
#appendix: App.x
#Abstract: Abs

In [ ]:
book_pattern = re.compile(r'0|[123]|(Abs)|(App)')
book_list = []
for para in treatise_paragraph_list:
    if book_pattern.match(para) != None:
        if book_pattern.match(para).group() not in book_list:
            book_list.append(book_pattern.match(para).group())
book_list

In [ ]:
new_para_list = []
for para in treatise_paragraph_list:
    if para[:3]=='Abs':
        old = para[3:]
        new = 'Abs.'+old
        new_para_list.append(new)
    else:
        new_para_list.append(para)
print(len(treatise_paragraph_list))        
print(len(new_para_list))

In [ ]:
blist = []
for para in new_para_list:
    sp = para.split('.')
    if sp[0] not in blist:
        blist.append(sp[0])
blist
        
    

In [ ]:
b_s_dict = {}
for book in blist:
    b_s_dict[book] = []
#the scope is off here. i'm doing this fresh from scratch with every paragraph...derp
for para in new_para_list:
    #create a list of sections with that book:
    sp = para.split('.')
    for book in blist:
        if sp[0] == book and sp[1] not in b_s_dict[book]:
            b_s_dict[book].append(sp[1])

In [ ]:
b_s_c_dict = {}
for book in blist:
    if book == "App" or book == "Abs" or book == "0":
        b_s_c_dict[book] = b_s_dict[book]
    else:
        b_s_c_dict[book] = {}
        for sect in b_s_dict[book]:
            b_s_c_dict[book][sect] = []
b_s_c_dict

In [ ]:
for para in new_para_list:
    sp = para.split('.')
    if len(sp) > 2:
        for book in b_s_c_dict.keys():
            if type(b_s_c_dict[book])==dict:
                for sect in b_s_c_dict[book].keys():
                    if (sp[0] == book and sp[1] == sect) and sp[2] not in b_s_c_dict[book][sect]:
                        b_s_c_dict[book][sect].append(sp[2])

In [ ]:
bscp = {}
for book in b_s_c_dict.keys():
    if book == "Abs" or book == "App" or book == "0":
        bscp[book] = b_s_c_dict[book]
    else:
        bscp[book] = {}
        for sect in b_s_c_dict[book].keys():
            bscp[book][sect] = {}
            for chapter in b_s_c_dict[book][sect]:
                bscp[book][sect][chapter] = []
                

In [ ]:
for para in new_para_list:
    sp = para.split('.')
    if len(sp) > 2:
        for book in bscp.keys():
            if type(bscp[book])==dict:
                for sect in bscp[book].keys():
                    for chap in bscp[book][sect].keys():
                        if (sp[0]==book and sp[1]==sect and sp[2]==chap) and sp[3] not in bscp[book][sect][chap]:
                            bscp[book][sect][chap].append(sp[3])

In [ ]:
import json

In [ ]:
jf = open("data/treatise_structure_dict.json", 'w')
json.dump(bscp, jf)
jf.close()


In [ ]:
testload = open('data/treatise_structure_dict.json', 'r')
td = json.load(testload)
testload.close()
td.keys()

In [ ]:
#okay now that i have this structure I can write the form selector page to generate the pages i'm interested in.
td['1']['4'].keys()